# 11 — Adaptive Covariance Tuning

**Repository:** `quantum-hardware-company`  
**Directory:** `control-stack/`

Notebook 11 replaces hand-tuned Kalman covariance assumptions with adaptive covariance estimates.

## Purpose

Previous notebooks used fixed values for:

```text
Q = process covariance
R = measurement covariance
q_cross = Ω/B coupling covariance
```

This notebook asks:

> Can the estimator adapt to changing noise and coupling regimes?

## Policies compared

- none
- moving average
- fixed scalar Kalman
- fixed joint Kalman
- adaptive R Kalman
- adaptive joint Kalman
- oracle

## Principle

Kalman performance depends on covariance calibration.

If the measurement environment changes, fixed covariance becomes stale.


## Adaptive covariance model

Use recent innovation history:

```text
innovation = measurement − prediction
```

Then update:

```text
R_adaptive ≈ rolling variance(innovation)
```

For joint Ω/B estimation:

```text
q_cross_adaptive ≈ rolling covariance(Ω innovation, B innovation)
```

This lets the estimator respond when:

- measurement noise increases
- Ω/B coupling changes
- fixed Q/R assumptions become inaccurate


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures/adaptive_covariance_tuning"
RESULTS_DIR = "results"
DOCS_DIR = "docs"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

def save_fig(name):
    plt.savefig(f"{FIG_DIR}/11_adaptive_covariance_tuning_{name}.png", dpi=300, bbox_inches="tight")
    plt.savefig(f"{FIG_DIR}/11_adaptive_covariance_tuning_{name}.pdf", bbox_inches="tight")


In [ ]:
def rabi_model(t, A, Omega, B):
    return A * np.sin(0.5 * Omega * t) ** 2 + B


def fit_model(model, x, y, p0, bounds=(-np.inf, np.inf)):
    params, cov = curve_fit(model, x, y, p0=p0, bounds=bounds, maxfev=30000)
    return params, cov


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def phase_lock_metric(signal, target):
    dot = np.dot(signal, target)
    norm = np.linalg.norm(signal) * np.linalg.norm(target)
    if norm == 0:
        return np.nan
    return dot / norm


def moving_average(x, window=4):
    y = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        y[i] = np.mean(x[lo:i + 1])
    return y


def kalman_filter_1d(z, q, r, x0=None, p0=1.0):
    z = np.asarray(z, dtype=float)
    x_hat = np.zeros_like(z)
    p_hist = np.zeros_like(z)
    k_hist = np.zeros_like(z)
    innovation = np.zeros_like(z)

    x = z[0] if x0 is None else float(x0)
    p = float(p0)

    for i, measurement in enumerate(z):
        x_pred = x
        p_pred = p + q

        innovation[i] = measurement - x_pred
        k_gain = p_pred / (p_pred + r)
        x = x_pred + k_gain * innovation[i]
        p = (1 - k_gain) * p_pred

        x_hat[i] = x
        p_hist[i] = p
        k_hist[i] = k_gain

    return x_hat, p_hist, k_hist, innovation


def adaptive_r_kalman_1d(z, q, r0, window=7, beta=0.35, r_floor=1e-7, r_ceiling=1e-2):
    z = np.asarray(z, dtype=float)
    x_hat = np.zeros_like(z)
    p_hist = np.zeros_like(z)
    k_hist = np.zeros_like(z)
    r_hist = np.zeros_like(z)
    innovation = np.zeros_like(z)

    x = z[0]
    p = 1.0
    r = r0
    buffer = []

    for i, measurement in enumerate(z):
        x_pred = x
        p_pred = p + q

        innov = measurement - x_pred
        innovation[i] = innov

        buffer.append(innov)
        if len(buffer) > window:
            buffer.pop(0)

        if len(buffer) >= 3:
            r_est = float(np.var(buffer))
            r_est = min(max(r_est, r_floor), r_ceiling)
            r = (1 - beta) * r + beta * r_est

        k_gain = p_pred / (p_pred + r)
        x = x_pred + k_gain * innov
        p = (1 - k_gain) * p_pred

        x_hat[i] = x
        p_hist[i] = p
        k_hist[i] = k_gain
        r_hist[i] = r

    return x_hat, p_hist, k_hist, r_hist, innovation


def joint_kalman(z, q_omega, q_b, q_cross, r_omega, r_b):
    z = np.asarray(z, dtype=float)
    n = z.shape[0]

    F = np.eye(2)
    H = np.eye(2)
    Q = np.array([[q_omega, q_cross], [q_cross, q_b]])
    R = np.array([[r_omega, 0.0], [0.0, r_b]])

    x = z[0].reshape(2, 1)
    P = np.eye(2)

    out = np.zeros((n, 2))
    innovations = np.zeros((n, 2))

    for i in range(n):
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q

        y = z[i].reshape(2, 1) - H @ x_pred
        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.inv(S)

        x = x_pred + K @ y
        P = (np.eye(2) - K @ H) @ P_pred

        out[i] = x.ravel()
        innovations[i] = y.ravel()

    return out, innovations


def adaptive_joint_kalman(z, q_omega0, q_b0, q_cross0, r_omega0, r_b0, window=8, beta=0.28):
    z = np.asarray(z, dtype=float)
    n = z.shape[0]

    F = np.eye(2)
    H = np.eye(2)

    x = z[0].reshape(2, 1)
    P = np.eye(2)

    out = np.zeros((n, 2))
    innovations = np.zeros((n, 2))
    r_hist = np.zeros((n, 2))
    q_cross_hist = np.zeros(n)

    r_omega = r_omega0
    r_b = r_b0
    q_cross = q_cross0

    buffer = []

    for i in range(n):
        Q = np.array([[q_omega0, q_cross], [q_cross, q_b0]])
        R = np.array([[r_omega, 0.0], [0.0, r_b]])

        x_pred = F @ x
        P_pred = F @ P @ F.T + Q

        y = z[i].reshape(2, 1) - H @ x_pred
        innov = y.ravel()
        innovations[i] = innov

        buffer.append(innov)
        if len(buffer) > window:
            buffer.pop(0)

        if len(buffer) >= 3:
            arr = np.array(buffer)
            cov = np.cov(arr.T)

            r_omega_est = min(max(float(cov[0, 0]), 1e-7), 1e-2)
            r_b_est = min(max(float(cov[1, 1]), 1e-7), 1e-2)

            q_cross_est = 0.25 * float(cov[0, 1])
            max_cross = 0.8 * np.sqrt(q_omega0 * q_b0)
            q_cross_est = float(np.clip(q_cross_est, -max_cross, max_cross))

            r_omega = (1 - beta) * r_omega + beta * r_omega_est
            r_b = (1 - beta) * r_b + beta * r_b_est
            q_cross = (1 - beta) * q_cross + beta * q_cross_est

        R = np.array([[r_omega, 0.0], [0.0, r_b]])
        Q = np.array([[q_omega0, q_cross], [q_cross, q_b0]])

        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.inv(S)

        x = x_pred + K @ y
        P = (np.eye(2) - K @ H) @ P_pred

        out[i] = x.ravel()
        r_hist[i] = [r_omega, r_b]
        q_cross_hist[i] = q_cross

    return out, innovations, r_hist, q_cross_hist


def evaluate_policy(name, delta_omega_hat, delta_b_hat, true_delta_omega, true_delta_b):
    omega_controlled = target_Omega + true_delta_omega - delta_omega_hat
    b_controlled = np.clip(target_B + true_delta_b - delta_b_hat, 0, 0.3)

    target_signal_local = rabi_model(pulse_t, target_A, target_Omega, target_B)

    response_rmse = []
    phase_lock = []
    controlled_signals = []

    for k in range(len(true_delta_omega)):
        y = rabi_model(pulse_t, target_A, omega_controlled[k], b_controlled[k])
        controlled_signals.append(y)
        response_rmse.append(rmse(y, target_signal_local))
        phase_lock.append(phase_lock_metric(y, target_signal_local))

    response_rmse = np.array(response_rmse)
    phase_lock = np.array(phase_lock)
    controlled_signals = np.array(controlled_signals)

    return {
        "name": name,
        "omega_command": np.asarray(delta_omega_hat),
        "b_command": np.asarray(delta_b_hat),
        "omega_rmse": rmse(omega_controlled, np.full(len(omega_controlled), target_Omega)),
        "b_rmse": rmse(b_controlled, np.full(len(b_controlled), target_B)),
        "response_rmse": response_rmse,
        "response_rmse_mean": float(np.mean(response_rmse)),
        "response_rmse_max": float(np.max(response_rmse)),
        "phase_lock": phase_lock,
        "min_phase_lock": float(np.min(phase_lock)),
        "mean_phase_lock": float(np.mean(phase_lock)),
        "max_angle_degrees": float(np.max(np.degrees(np.arccos(np.clip(phase_lock, -1, 1))))),
        "all_blocks_phase_locked": bool(np.all(phase_lock >= 1 / np.sqrt(2))),
        "controlled_signals": controlled_signals,
    }


## Simulate nonstationary covariance environment

This environment has:

- weak coupling early
- increased measurement noise mid-run
- stronger Ω/B coupling late-run

This makes fixed covariance assumptions stale.


In [ ]:
n_blocks = 72
block_times = np.arange(n_blocks)

pulse_t = np.linspace(0, 10, 140)

target_A = 0.88
target_Omega = 2.45
target_B = 0.06
target_signal = rabi_model(pulse_t, target_A, target_Omega, target_B)

coupling_profile = np.piecewise(
    block_times,
    [block_times < 24, (block_times >= 24) & (block_times < 48), block_times >= 48],
    [0.12, 0.25, 0.42],
)

shared = 0.018 * np.sin(2 * np.pi * block_times / 29 + 0.2)
omega_private = (
    0.040 * np.sin(2 * np.pi * block_times / 23 + 0.35)
    + 0.012 * np.sin(2 * np.pi * block_times / 8)
)
b_private = (
    0.0008 * block_times
    + 0.004 * np.sin(2 * np.pi * block_times / 15 + 0.4)
)

true_delta_Omega = omega_private + shared
true_delta_B = b_private + coupling_profile * shared

Omega_true = target_Omega + true_delta_Omega
B_true = target_B + true_delta_B
A_true = target_A + 0.010 * np.sin(2 * np.pi * block_times / 19)

sigma_profile = np.piecewise(
    block_times,
    [block_times < 24, (block_times >= 24) & (block_times < 48), block_times >= 48],
    [0.035, 0.060, 0.050],
)

print(f"n_blocks = {n_blocks}")
print(f"early coupling = {coupling_profile[0]:.2f}")
print(f"mid coupling   = {coupling_profile[30]:.2f}")
print(f"late coupling  = {coupling_profile[-1]:.2f}")


## Generate calibration data and fits


In [ ]:
Y_obs = []
Y_clean = []

for k in range(n_blocks):
    y_clean = rabi_model(pulse_t, A_true[k], Omega_true[k], B_true[k])
    y_obs = np.clip(y_clean + sigma_profile[k] * np.random.randn(len(pulse_t)), 0, 1)
    Y_clean.append(y_clean)
    Y_obs.append(y_obs)

Y_obs = np.array(Y_obs)
Y_clean = np.array(Y_clean)

fit_params = []
fit_stds = []

initial_guess = [0.85, 2.40, 0.05]
bounds = ([0.0, 0.0, 0.0], [1.2, 5.0, 0.3])

for k in range(n_blocks):
    params, cov = fit_model(rabi_model, pulse_t, Y_obs[k], p0=initial_guess, bounds=bounds)
    fit_params.append(params)
    fit_stds.append(np.sqrt(np.diag(cov)))

fit_params = np.array(fit_params)
fit_stds = np.array(fit_stds)

delta_Omega_est = fit_params[:, 1] - target_Omega
delta_B_est = fit_params[:, 2] - target_B

print("Calibration fits complete.")
print(f"measured Ω/B correlation = {np.corrcoef(delta_Omega_est, delta_B_est)[0,1]:.3f}")


## Define fixed and adaptive policies


In [ ]:
delta_Omega_none = np.zeros(n_blocks)
delta_B_none = np.zeros(n_blocks)

delta_Omega_ma = moving_average(delta_Omega_est, window=4)
delta_B_ma = moving_average(delta_B_est, window=4)

r_omega0 = float(np.median(fit_stds[:, 1] ** 2))
r_b0 = float(np.median(fit_stds[:, 2] ** 2))

q_omega = 8e-4
q_b = 1e-5

delta_Omega_scalar, _, _, innov_omega_scalar = kalman_filter_1d(delta_Omega_est, q=q_omega, r=r_omega0)
delta_B_scalar, _, _, innov_b_scalar = kalman_filter_1d(delta_B_est, q=q_b, r=r_b0)

q_cross_fixed = 0.25 * np.sqrt(q_omega * q_b)

joint_fixed, innov_joint_fixed = joint_kalman(
    np.column_stack([delta_Omega_est, delta_B_est]),
    q_omega=q_omega,
    q_b=q_b,
    q_cross=q_cross_fixed,
    r_omega=r_omega0,
    r_b=r_b0,
)

delta_Omega_joint = joint_fixed[:, 0]
delta_B_joint = joint_fixed[:, 1]

delta_Omega_adapt_r, _, _, r_omega_hist, innov_omega_adapt = adaptive_r_kalman_1d(
    delta_Omega_est, q=q_omega, r0=r_omega0, window=7, beta=0.35
)
delta_B_adapt_r, _, _, r_b_hist, innov_b_adapt = adaptive_r_kalman_1d(
    delta_B_est, q=q_b, r0=r_b0, window=7, beta=0.35
)

joint_adaptive, innov_adaptive, r_hist_joint, q_cross_hist = adaptive_joint_kalman(
    np.column_stack([delta_Omega_est, delta_B_est]),
    q_omega0=q_omega,
    q_b0=q_b,
    q_cross0=q_cross_fixed,
    r_omega0=r_omega0,
    r_b0=r_b0,
    window=8,
    beta=0.28,
)

delta_Omega_adapt_joint = joint_adaptive[:, 0]
delta_B_adapt_joint = joint_adaptive[:, 1]

delta_Omega_oracle = true_delta_Omega.copy()
delta_B_oracle = true_delta_B.copy()

print("Policies ready.")
print(f"r_omega0 = {r_omega0:.3e}")
print(f"r_b0     = {r_b0:.3e}")
print(f"q_cross_fixed = {q_cross_fixed:.3e}")


## Adaptive measurement covariance traces


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, r_omega_hist, label="adaptive R Ω")
plt.plot(block_times, r_b_hist, label="adaptive R B")
plt.axvspan(24, 47, alpha=0.12, label="higher noise region")
plt.xlabel("calibration block")
plt.ylabel("adaptive R estimate")
plt.title("Adaptive covariance tuning: measurement covariance traces")
plt.legend()
plt.tight_layout()
save_fig("adaptive_r_traces")
plt.show()


## Adaptive coupling trace


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, q_cross_hist, label="adaptive q_cross")
plt.axhline(q_cross_fixed, linestyle="--", label="fixed q_cross")
plt.xlabel("calibration block")
plt.ylabel("q_cross")
plt.title("Adaptive covariance tuning: joint coupling trace")
plt.legend()
plt.tight_layout()
save_fig("adaptive_coupling_trace")
plt.show()


## Ω tracking


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, true_delta_Omega, linewidth=2, label="true Ω drift")
plt.plot(block_times, delta_Omega_est, marker="o", linewidth=1, label="measured Ω")
plt.plot(block_times, delta_Omega_scalar, label="fixed scalar Kalman")
plt.plot(block_times, delta_Omega_joint, label="fixed joint Kalman")
plt.plot(block_times, delta_Omega_adapt_joint, label="adaptive joint Kalman")
plt.xlabel("calibration block")
plt.ylabel("Ω drift")
plt.title("Adaptive covariance tuning: Ω tracking")
plt.legend()
plt.tight_layout()
save_fig("omega_tracking")
plt.show()


## B tracking


In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(block_times, true_delta_B, linewidth=2, label="true B drift")
plt.plot(block_times, delta_B_est, marker="o", linewidth=1, label="measured B")
plt.plot(block_times, delta_B_scalar, label="fixed scalar Kalman")
plt.plot(block_times, delta_B_joint, label="fixed joint Kalman")
plt.plot(block_times, delta_B_adapt_joint, label="adaptive joint Kalman")
plt.xlabel("calibration block")
plt.ylabel("B drift")
plt.title("Adaptive covariance tuning: B tracking")
plt.legend()
plt.tight_layout()
save_fig("offset_tracking")
plt.show()


## Evaluate policies


In [ ]:
policies = {
    "none": (delta_Omega_none, delta_B_none),
    "moving_average": (delta_Omega_ma, delta_B_ma),
    "fixed_scalar_kalman": (delta_Omega_scalar, delta_B_scalar),
    "fixed_joint_kalman": (delta_Omega_joint, delta_B_joint),
    "adaptive_r_kalman": (delta_Omega_adapt_r, delta_B_adapt_r),
    "adaptive_joint_kalman": (delta_Omega_adapt_joint, delta_B_adapt_joint),
    "oracle": (delta_Omega_oracle, delta_B_oracle),
}

policy_results = {}

for name, (om_hat, b_hat) in policies.items():
    policy_results[name] = evaluate_policy(name, om_hat, b_hat, true_delta_Omega, true_delta_B)

for name, result in policy_results.items():
    print(
        f"{name:24s} | "
        f"mean RMSE = {result['response_rmse_mean']:.6f} | "
        f"max = {result['response_rmse_max']:.6f} | "
        f"min cos = {result['min_phase_lock']:.6f}"
    )


## Response RMSE comparison


In [ ]:
plt.figure(figsize=(11, 5))
for name, result in policy_results.items():
    plt.plot(block_times, result["response_rmse"], marker="o", linewidth=1, label=name)

plt.axvspan(24, 47, alpha=0.12, label="higher noise region")
plt.xlabel("calibration block")
plt.ylabel("RMSE vs target response")
plt.title("Adaptive covariance tuning: response RMSE comparison")
plt.legend()
plt.tight_layout()
save_fig("response_rmse_comparison")
plt.show()


## CGCS phase-lock stability


In [ ]:
threshold = 1 / np.sqrt(2)

plt.figure(figsize=(11, 5))
plt.axhline(threshold, linestyle="--", linewidth=1, label="45° threshold")

for name, result in policy_results.items():
    plt.plot(block_times, result["phase_lock"], marker="o", linewidth=1, label=name)

plt.ylim(0.68, 1.01)
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.title("Adaptive covariance tuning: CGCS phase-lock stability")
plt.legend()
plt.tight_layout()
save_fig("cgcs_stability_comparison")
plt.show()


## Policy ranking


In [ ]:
ranking = sorted(policy_results.values(), key=lambda r: r["response_rmse_mean"])

policy_table = []

for rank, result in enumerate(ranking, start=1):
    policy_table.append({
        "rank": rank,
        "policy": result["name"],
        "omega_rmse": result["omega_rmse"],
        "b_rmse": result["b_rmse"],
        "response_rmse_mean": result["response_rmse_mean"],
        "response_rmse_max": result["response_rmse_max"],
        "min_phase_lock": result["min_phase_lock"],
        "mean_phase_lock": result["mean_phase_lock"],
        "max_angle_degrees": result["max_angle_degrees"],
        "all_blocks_phase_locked": result["all_blocks_phase_locked"],
    })

policy_table


In [ ]:
names = [r["policy"] for r in policy_table]
means = [r["response_rmse_mean"] for r in policy_table]
maxes = [r["response_rmse_max"] for r in policy_table]

x = np.arange(len(names))
width = 0.35

plt.figure(figsize=(11, 5))
plt.bar(x - width/2, means, width, label="mean RMSE")
plt.bar(x + width/2, maxes, width, label="max RMSE")
plt.xticks(x, names, rotation=25, ha="right")
plt.ylabel("response RMSE")
plt.title("Adaptive covariance tuning: policy ranking")
plt.legend()
plt.tight_layout()
save_fig("policy_ranking_summary")
plt.show()


## Adaptation window sweep


In [ ]:
windows = [3, 5, 7, 9, 12, 16]
window_rmse = []

for w in windows:
    joint_tmp, _, _, _ = adaptive_joint_kalman(
        np.column_stack([delta_Omega_est, delta_B_est]),
        q_omega0=q_omega,
        q_b0=q_b,
        q_cross0=q_cross_fixed,
        r_omega0=r_omega0,
        r_b0=r_b0,
        window=w,
        beta=0.28,
    )

    ev = evaluate_policy(
        f"window_{w}",
        joint_tmp[:, 0],
        joint_tmp[:, 1],
        true_delta_Omega,
        true_delta_B,
    )

    window_rmse.append(ev["response_rmse_mean"])

window_rmse = np.array(window_rmse)
best_idx = int(np.argmin(window_rmse))
best_window = int(windows[best_idx])

print(f"best window = {best_window}")
print(f"best window RMSE = {window_rmse[best_idx]:.6f}")


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(windows, window_rmse, marker="o")
plt.axvline(best_window, linestyle=":", label=f"best window = {best_window}")
plt.xlabel("adaptive covariance window")
plt.ylabel("mean response RMSE")
plt.title("Adaptive covariance tuning: window sweep")
plt.legend()
plt.tight_layout()
save_fig("adaptation_window_sweep")
plt.show()


## Worst-case block comparison


In [ ]:
worst_block = int(np.argmax(policy_results["none"]["response_rmse"]))

plt.figure(figsize=(11, 5))
plt.plot(pulse_t, target_signal, linewidth=3, label="target")

for name in ["none", "fixed_joint_kalman", "adaptive_joint_kalman", "oracle"]:
    plt.plot(
        pulse_t,
        policy_results[name]["controlled_signals"][worst_block],
        linewidth=2,
        label=name,
    )

plt.xlabel("pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Adaptive covariance tuning: worst-case block {worst_block}")
plt.legend()
plt.tight_layout()
save_fig("worst_case_block_comparison")
plt.show()


## Save summary JSON


In [ ]:
summary = {
    "notebook": "11_adaptive_covariance_tuning",
    "blocks": int(n_blocks),
    "phase_lock_threshold": float(threshold),
    "regimes": {
        "early": "low noise / weak coupling",
        "middle": "higher measurement noise",
        "late": "stronger coupling",
    },
    "fixed_covariance": {
        "q_omega": float(q_omega),
        "q_b": float(q_b),
        "q_cross_fixed": float(q_cross_fixed),
        "r_omega0": float(r_omega0),
        "r_b0": float(r_b0),
    },
    "adaptive_covariance": {
        "final_r_omega": float(r_omega_hist[-1]),
        "final_r_b": float(r_b_hist[-1]),
        "final_q_cross": float(q_cross_hist[-1]),
        "best_window": int(best_window),
        "best_window_response_rmse": float(window_rmse[best_idx]),
    },
    "ranking": policy_table,
}

with open(f"{RESULTS_DIR}/adaptive_covariance_tuning_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/adaptive_covariance_tuning_summary.json")


## Write docs and results markdown


In [ ]:
docs_md = '''# Adaptive Covariance Tuning

Adaptive Kalman covariance tuning for nonstationary measurement noise and changing Ω/B coupling.

---

## Pipeline

calibration → innovation tracking → adaptive covariance update → stabilized response

---

## Figures

### Adaptive R traces

![Adaptive R](../figures/adaptive_covariance_tuning/11_adaptive_covariance_tuning_adaptive_r_traces.png)

Adaptive measurement covariance responds to changing noise regimes.

---

### Adaptive coupling trace

![Adaptive coupling](../figures/adaptive_covariance_tuning/11_adaptive_covariance_tuning_adaptive_coupling_trace.png)

Adaptive q_cross tracks changing Ω/B coupling structure.

---

### Ω tracking

![Omega tracking](../figures/adaptive_covariance_tuning/11_adaptive_covariance_tuning_omega_tracking.png)

Adaptive joint Kalman tracks Ω under changing covariance conditions.

---

### B tracking

![Offset tracking](../figures/adaptive_covariance_tuning/11_adaptive_covariance_tuning_offset_tracking.png)

Adaptive joint Kalman stabilizes B tracking across regimes.

---

### Response RMSE comparison

![Response RMSE](../figures/adaptive_covariance_tuning/11_adaptive_covariance_tuning_response_rmse_comparison.png)

Response-level RMSE compares fixed and adaptive covariance policies.

---

### CGCS phase-lock stability

![CGCS stability](../figures/adaptive_covariance_tuning/11_adaptive_covariance_tuning_cgcs_stability_comparison.png)

All policies are evaluated against the 45° CGCS threshold.

---

### Policy ranking

![Policy ranking](../figures/adaptive_covariance_tuning/11_adaptive_covariance_tuning_policy_ranking_summary.png)

Ranking summarizes mean and worst-case response error.

---

### Adaptation window sweep

![Window sweep](../figures/adaptive_covariance_tuning/11_adaptive_covariance_tuning_adaptation_window_sweep.png)

Window sweep tests adaptation speed vs noise sensitivity.

---

### Worst-case block comparison

![Worst case](../figures/adaptive_covariance_tuning/11_adaptive_covariance_tuning_worst_case_block_comparison.png)

Worst-case response compares fixed and adaptive estimators.

---

## Key Takeaway

Adaptive covariance tuning lets the estimator respond to nonstationary noise and changing Ω/B coupling instead of relying on fixed Q/R assumptions.
'''

summary_rows = "\n".join(
    [
        f"| {r['rank']} | {r['policy']} | {r['response_rmse_mean']:.6f} | {r['response_rmse_max']:.6f} |"
        for r in policy_table
    ]
)

summary_md = f'''# Adaptive Covariance Tuning Results Summary

---

## Configuration

- Blocks: {n_blocks}
- Phase-lock threshold: {threshold:.4f}
- Best adaptation window: {best_window}
- Best window RMSE: {window_rmse[best_idx]:.6f}

---

## Fixed Covariance

- q_omega: {q_omega:.3e}
- q_b: {q_b:.3e}
- q_cross_fixed: {q_cross_fixed:.3e}
- r_omega0: {r_omega0:.3e}
- r_b0: {r_b0:.3e}

---

## Adaptive Covariance

- final_r_omega: {r_omega_hist[-1]:.3e}
- final_r_b: {r_b_hist[-1]:.3e}
- final_q_cross: {q_cross_hist[-1]:.3e}

---

## Policy Ranking

| Rank | Policy | Mean RMSE | Max RMSE |
|------|--------|----------:|---------:|
{summary_rows}

---

## Key Insight

Adaptive covariance tuning tests when fixed Kalman assumptions become stale under changing noise and coupling regimes.
'''

with open(f"{DOCS_DIR}/adaptive_covariance_tuning.md", "w") as f:
    f.write(docs_md)

with open(f"{RESULTS_DIR}/adaptive_covariance_tuning_summary.md", "w") as f:
    f.write(summary_md)

print(f"Wrote {DOCS_DIR}/adaptive_covariance_tuning.md")
print(f"Wrote {RESULTS_DIR}/adaptive_covariance_tuning_summary.md")


## Zip outputs for download from Colab


In [ ]:
!zip -r adaptive_covariance_tuning_outputs.zip figures results docs


## Control-stack status

Notebook 11 adds adaptive covariance tuning.

Next:

```text
12_failure_boundary_phase_lock.ipynb
```

Goal:

> push controllers toward the CGCS 45° boundary and measure failure modes.
